In [1]:
# ============================================
#  Bibliotecas padrão
# ============================================
import os
import re
import random
import string
import emoji
import psutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import randint
import shap


# ============================================
#  NLP e Pré-processamento
# ============================================
import nltk
from nltk.corpus import stopwords
import spacy
from scipy.sparse import csr_matrix, hstack



# ============================================
#  Extração de features (Vetorização)
# ============================================
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

# ============================================
#  Modelos de Machine Learning
# ============================================
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.naive_bayes import BernoulliNB, MultinomialNB, ComplementNB
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC, LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.calibration import CalibratedClassifierCV

# ============================================
#  Validação, divisão e utilidades
# ============================================
from sklearn.model_selection import (
    train_test_split,
    cross_validate,
    learning_curve,
    StratifiedKFold
)

# ============================================
#  Métricas e relatórios
# ============================================
from sklearn import metrics
from sklearn.metrics import ( 
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

# ============================================
#  Interpretabilidade (SHAP e LIME)
# ============================================
import shap
import lime
from lime.lime_text import LimeTextExplainer
from sklearn.pipeline import make_pipeline

# ============================================
#  Configuração de visualização
# ============================================
get_ipython().run_line_magic('matplotlib', 'inline')
sns.set(style="darkgrid")

# ============================================
#  Configuração de desempenho
# ============================================
os.environ["LOKY_MAX_CPU_COUNT"] = str(psutil.cpu_count(logical=False))


d:\Arquivos_Acer\Documents\UFC\PrejudicePT-br-main\ambiente_prejudice\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
filepath = '../../Datasets/Telegram_Tratado_Rotulado_Revisado_Final.csv'

df = pd.read_csv(filepath)    


# Funções que serão usadas para processamento de texto

In [3]:
unicode_emoji = {}
for key, value in emoji.EMOJI_DATA.items():
    try:
        unicode_emoji[key] = value['pt']
    except:
        pass

#emojis and punctuation
emojis_list = list(unicode_emoji)
punct = list(string.punctuation)
emojis_punct = emojis_list + punct

def processEmojisPunctuation(text, remove_punct = True):
    '''
    Put spaces between emojis. Removes punctuation.
    '''
    #get all unique chars
    chars = set(text)
    #for each unique char in text, do:
    for c in chars:
        #remove punctuation
        if remove_punct:
            if c in emojis_list:
                text = text.replace(c, ' ' + c + ' ')
            if c in punct:
                text = text.replace(c, ' ')

        #put spaces between punctuation
        else:
            if c in emojis_punct:
                text = text.replace(c, ' ' + c + ' ')          

    text = text.replace('  ', ' ')
    return text

#stop words removal
stop_words = list(stopwords.words('portuguese'))
new_stopwords = ['aí','pra','vão','vou','onde','lá','aqui',
                 'tá','pode','pois','so','deu','agora','todo',
                 'nao','ja','vc', 'bom', 'ai','kkk','kkkk','ta', 'voce', 'alguem', 'ne', 'pq',
                 'cara','to','mim','la','vcs','tbm', 'tudo']
stop_words = stop_words + new_stopwords
final_stop_words = []
for sw in stop_words:
    sw = ' '+ sw + ' '
    final_stop_words.append(sw)

def removeStopwords(text):
    for sw in final_stop_words:
        text = text.replace(sw,' ')
    text = text.replace('  ',' ')
    return text

#lemmatization
nlp = spacy.load('pt_core_news_sm')
def lemmatization(text):
    doc = nlp(text)
    for token in doc:
        if token.text != token.lemma_:
            text = text.replace(token.text, token.lemma_)
    return text


def domainUrl(text):
    '''
    Substitutes an URL in a text for the domain of this URL
    Input: an string
    Output: the string with the modified URL
    '''    
    if 'http' in text:
        re_url = '[^\s]*https*://[^\s]*'
        matches = re.findall(re_url, text, flags=re.IGNORECASE)
        for m in matches:
            domain = m.split('//')
            domain = domain[1].split('/')[0]
            text = re.sub(re_url, domain, text, 1)
        return text
    else:
        return text 

def preprocess(text):
    text = text.lower().strip()
    text = domainUrl(text)
    text = processEmojisPunctuation(text)
    text = removeStopwords(text)
    text = lemmatization(text)
    return text


In [4]:
# Função de pré-processamento
def preprocess_data(df, experiment):
    if 'processed' in experiment:
        print("Pré-processamento ativado.")
        pro_texts = [preprocess(t) for t in df['text_content_anonymous']]
    else:
        print("Sem pré-processamento.")
        pro_texts = [processEmojisPunctuation(t.lower(), remove_punct=False) for t in df['text_content_anonymous']]
    return pro_texts


In [5]:

# ===============================
# 1. Funções utilitárias
# ===============================

def carregar_dicionario_personalizado(dic_path):
    categorias = {}
    lexicon = {}
    dentro_das_categorias = False

    with open(dic_path, 'r', encoding='utf-8') as file:
        for linha in file:
            linha = linha.strip()
            if linha == '%':
                dentro_das_categorias = not dentro_das_categorias
                continue
            if dentro_das_categorias:
                codigo, categoria = linha.split()
                categorias[codigo] = categoria
            else:
                partes = linha.split("\t")
                palavra = partes[0]
                categoria_ids = partes[1:]
                lexicon[palavra] = [categorias[codigo] for codigo in categoria_ids if codigo in categorias]
    
    return lexicon, list(categorias.values())


def tokenize(text):
    """Tokenização simples."""
    return re.findall(r"\w+", text.lower(), re.UNICODE)


def gerar_features_dicionario(textos, lexicon):
    """Gera matriz esparsa de contagem de palavras do dicionário."""
    palavras = list(lexicon.keys())
    idx_map = {palavra: i for i, palavra in enumerate(palavras)}
    features = np.zeros((len(textos), len(palavras)), dtype=int)
    
    for i, texto in enumerate(textos):
        for token in tokenize(texto):
            if token in idx_map:
                features[i, idx_map[token]] += 1
    
    return csr_matrix(features), palavras


# ===============================
# 2. Carregar dicionário e dados
# ===============================

dic_path = '../../Dicionário/v2_SocialLIWC_formatado_ordenado.dic'
lexicon, category_names = carregar_dicionario_personalizado(dic_path)
print("Dicionário carregado com", len(lexicon), "palavras e", len(category_names), "categorias.")


Dicionário carregado com 842 palavras e 9 categorias.


## Experimento IA Explicável - Sem pré processamento

In [6]:
def preprocess_data(data, experiment):
    
    # aceitar lista ou dataframe
    if isinstance(data, list):
        texts = data
    else:
        texts = list(data['text_content_anonymous'])

    if experiment == "ml-dic-only":
        print("Sem pré-processamento.")
        pro_texts = [processEmojisPunctuation(t.lower(), remove_punct=False) for t in texts]
    else:
        print("Pré-processamento padrão.")
        pro_texts = [processEmojisPunctuation(t.lower(), remove_punct=True) for t in texts]

    return pro_texts


### SHAP - LinearSVC

In [13]:


# ==========================================
# 1. Preparação dos dados
# ==========================================

experiment = 'ml-dic-mnb'
print(f"Rodando experimento: {experiment}")

df = df.drop_duplicates(subset=['text_content_anonymous'])
df['text_content_anonymous'] = df['text_content_anonymous'].fillna("").astype(str)

pro_texts = preprocess_data(df, experiment)
y = df['preconceito']

print(f"Tamanho total: {len(y)} | Classe 1: {sum(y)} | Classe 0: {len(y) - sum(y)}")


# ==========================================
# 2. Validação cruzada
# ==========================================

kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
accs, precs, recs, f1s, aucs = [], [], [], [], []

fold = 1
last_model = None
last_X_test = None
feature_names = list(lexicon.keys())

for train_idx, test_idx in kf.split(pro_texts, y):
    X_train_texts = [pro_texts[i] for i in train_idx]
    X_test_texts = [pro_texts[i] for i in test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    # Gerar features do dicionário (binárias ou contagem)
    X_train, feature_names = gerar_features_dicionario(X_train_texts, lexicon)
    X_test, _ = gerar_features_dicionario(X_test_texts, lexicon)

    # === Modelo Multinomial Naive Bayes ===
    model = MultinomialNB()
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    accs.append(accuracy_score(y_test, y_pred))
    precs.append(precision_score(y_test, y_pred))
    recs.append(recall_score(y_test, y_pred))
    f1s.append(f1_score(y_test, y_pred))
    aucs.append(roc_auc_score(y_test, y_prob))

    print(f"\n=== Fold {fold} ===")
    print(f"Acurácia: {accs[-1]:.3f} | Precisão: {precs[-1]:.3f} | Revocação: {recs[-1]:.3f} | "
          f"F1: {f1s[-1]:.3f} | AUC: {aucs[-1]:.3f}")

    last_model = model
    last_X_test = X_test
    fold += 1


# ==========================================
# 3. Resultados médios
# ==========================================

print("\n=== Resultados Médios (5 Folds) ===")
print(f"Acurácia média: {np.mean(accs):.3f} ± {np.std(accs):.3f}")
print(f"Precisão média: {np.mean(precs):.3f}")
print(f"Revocação média: {np.mean(recs):.3f}")
print(f"F1 média:        {np.mean(f1s):.3f}")
print(f"ROC AUC média:   {np.mean(aucs):.3f}")


# ==========================================
# 4. Explicabilidade local (MultinomialNB + dicionário)
# ==========================================

print("\n=== SHAP LOCAL (MultinomialNB com Dicionário) ===")

# Converter sparse -> dense
X_dense = last_X_test.toarray()
feature_names = np.array(feature_names)

# Escolher uma amostra com pelo menos 1 palavra
nonzero_counts = np.count_nonzero(X_dense, axis=1)
valid_idxs = np.where(nonzero_counts > 0)[0]
if len(valid_idxs) == 0:
    raise ValueError("Nenhuma amostra contém palavras do dicionário.")

idx_local = np.random.choice(valid_idxs)
X_text_local = X_dense[idx_local:idx_local+1]
print(f"\nAmostra escolhida: índice {idx_local} (contém {nonzero_counts[idx_local]} palavras do dicionário)")


# ==== 1. SHAP manual (coeficientes log-probabilidade) ====
# Para MultinomialNB, feature_log_prob_ representa log(P(feature|classe))
# A diferença entre classes fornece uma medida da importância

log_probs = last_model.feature_log_prob_
coef = (log_probs[1] - log_probs[0])  # diferença entre classe 1 e classe 0
intercept = last_model.class_log_prior_[1] - last_model.class_log_prior_[0]

vals = X_text_local.flatten() * coef

df_local = pd.DataFrame({
    "Palavra": feature_names,
    "Valor": X_text_local.flatten(),
    "SHAP Value": vals
})
df_local["Abs"] = np.abs(df_local["SHAP Value"])
df_local = df_local[df_local["Valor"] > 0].sort_values("Abs", ascending=False)

# Top 10 positivas e negativas
df_pos = df_local[df_local["SHAP Value"] > 0].head(10)
df_neg = df_local[df_local["SHAP Value"] < 0].head(10)

print("\n=== Top 10 palavras explicativas para PRECONCEITO ===")
print(df_pos[["Palavra", "SHAP Value"]].to_string(index=False))

print("\n=== Top 10 palavras explicativas para NÃO PRECONCEITO ===")
print(df_neg[["Palavra", "SHAP Value"]].to_string(index=False))


# ==========================================
# 5. Fidelidade (Comprehensiveness / Sufficiency)
# ==========================================

tokens_exp = df_local.head(10)["Palavra"].tolist()

def compute_fidelity_nb(text, tokens_exp, model, lexicon):
    tokens_text = preprocess_data([text], experiment)[0].split()
    tokens_exp = [t for t in tokens_exp if t in tokens_text]

    palavras = list(lexicon.keys())
    idx_map = {p: i for i, p in enumerate(palavras)}

    def vectorize_tokens(tokens):
        features = np.zeros((1, len(palavras)))
        for t in tokens:
            if t in idx_map:
                features[0, idx_map[t]] += 1
        return csr_matrix(features)

    def prob_from_tokens(tokens):
        return model.predict_proba(vectorize_tokens(tokens))[0][1]

    p_original = prob_from_tokens(tokens_text)
    tokens_removed = [t for t in tokens_text if t not in tokens_exp]
    p_removed = prob_from_tokens(tokens_removed)

    p_only = prob_from_tokens(tokens_exp) if tokens_exp else 0.0

    comp = (p_original - p_removed) / max(p_original, 1e-6)
    suff = (p_original - p_only) / max(p_original, 1e-6)

    return comp, suff, p_original, p_removed, p_only


text_example = df.iloc[idx_local]["text_content_anonymous"]
comp, suff, p_o, p_r, p_s = compute_fidelity_nb(text_example, tokens_exp, last_model, lexicon)

print("\n=== Fidelidade (via MultinomialNB + Dicionário) ===")
print(f"Palavras explicativas: {tokens_exp}")
print(f"Probabilidade original: {p_o:.3f}")
print(f"Prob. sem palavras explicativas: {p_r:.3f}")
print(f"Prob. só com palavras explicativas: {p_s:.3f}")
print(f"Abrangência (Comprehensiveness): {comp:.3f}")
print(f"Suficiência (Sufficiency):       {suff:.3f}")


Rodando experimento: ml-dic-mnb
Pré-processamento padrão.
Tamanho total: 2997 | Classe 1: 1497 | Classe 0: 1500

=== Fold 1 ===
Acurácia: 0.723 | Precisão: 0.789 | Revocação: 0.610 | F1: 0.688 | AUC: 0.798

=== Fold 2 ===
Acurácia: 0.762 | Precisão: 0.840 | Revocação: 0.647 | F1: 0.731 | AUC: 0.827

=== Fold 3 ===
Acurácia: 0.721 | Precisão: 0.743 | Revocação: 0.676 | F1: 0.708 | AUC: 0.803

=== Fold 4 ===
Acurácia: 0.726 | Precisão: 0.773 | Revocação: 0.639 | F1: 0.700 | AUC: 0.795

=== Fold 5 ===
Acurácia: 0.743 | Precisão: 0.776 | Revocação: 0.682 | F1: 0.726 | AUC: 0.820

=== Resultados Médios (5 Folds) ===
Acurácia média: 0.735 ± 0.015
Precisão média: 0.784
Revocação média: 0.651
F1 média:        0.710
ROC AUC média:   0.809

=== SHAP LOCAL (MultinomialNB com Dicionário) ===

Amostra escolhida: índice 572 (contém 2 palavras do dicionário)

=== Top 10 palavras explicativas para PRECONCEITO ===
Palavra  SHAP Value
 ladrão    4.712457

=== Top 10 palavras explicativas para NÃO PRECON

In [15]:


# ==========================================
# 1. Preparação dos dados
# ==========================================

experiment = 'ml-dic-mnb'
print(f"Rodando experimento: {experiment}")

df = df.drop_duplicates(subset=['text_content_anonymous'])
df['text_content_anonymous'] = df['text_content_anonymous'].fillna("").astype(str)

pro_texts = preprocess_data(df, experiment)
y = df['preconceito']

print(f"Tamanho total: {len(y)} | Classe 1: {sum(y)} | Classe 0: {len(y) - sum(y)}")


# ==========================================
# 2. Validação cruzada
# ==========================================

kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
accs, precs, recs, f1s, aucs = [], [], [], [], []

fold = 1
last_model = None
last_X_test = None
feature_names = list(lexicon.keys())

for train_idx, test_idx in kf.split(pro_texts, y):
    X_train_texts = [pro_texts[i] for i in train_idx]
    X_test_texts = [pro_texts[i] for i in test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    # Gerar features do dicionário (binárias ou contagem)
    X_train, feature_names = gerar_features_dicionario(X_train_texts, lexicon)
    X_test, _ = gerar_features_dicionario(X_test_texts, lexicon)

    # === Modelo Multinomial Naive Bayes ===
    model = MultinomialNB()
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    accs.append(accuracy_score(y_test, y_pred))
    precs.append(precision_score(y_test, y_pred))
    recs.append(recall_score(y_test, y_pred))
    f1s.append(f1_score(y_test, y_pred))
    aucs.append(roc_auc_score(y_test, y_prob))

    print(f"\n=== Fold {fold} ===")
    print(f"Acurácia: {accs[-1]:.3f} | Precisão: {precs[-1]:.3f} | Revocação: {recs[-1]:.3f} | "
          f"F1: {f1s[-1]:.3f} | AUC: {aucs[-1]:.3f}")

    last_model = model
    last_X_test = X_test
    fold += 1


# ==========================================
# 3. Resultados médios
# ==========================================

print("\n=== Resultados Médios (5 Folds) ===")
print(f"Acurácia média: {np.mean(accs):.3f} ± {np.std(accs):.3f}")
print(f"Precisão média: {np.mean(precs):.3f}")
print(f"Revocação média: {np.mean(recs):.3f}")
print(f"F1 média:        {np.mean(f1s):.3f}")
print(f"ROC AUC média:   {np.mean(aucs):.3f}")


# ==========================================
# 4. Explicabilidade local (MultinomialNB + dicionário)
# ==========================================

print("\n=== SHAP LOCAL (MultinomialNB com Dicionário) ===")

# Converter sparse -> dense
X_dense = last_X_test.toarray()
feature_names = np.array(feature_names)

# Escolher uma amostra com pelo menos 1 palavra
nonzero_counts = np.count_nonzero(X_dense, axis=1)
valid_idxs = np.where(nonzero_counts > 0)[0]
if len(valid_idxs) == 0:
    raise ValueError("Nenhuma amostra contém palavras do dicionário.")

idx_local = np.random.choice(valid_idxs)
X_text_local = X_dense[idx_local:idx_local+1]
print(f"\nAmostra escolhida: índice {idx_local} (contém {nonzero_counts[idx_local]} palavras do dicionário)")


# ==== 1. SHAP manual (coeficientes log-probabilidade) ====
# Para MultinomialNB, feature_log_prob_ representa log(P(feature|classe))
# A diferença entre classes fornece uma medida da importância

log_probs = last_model.feature_log_prob_
coef = (log_probs[1] - log_probs[0])  # diferença entre classe 1 e classe 0
intercept = last_model.class_log_prior_[1] - last_model.class_log_prior_[0]

vals = X_text_local.flatten() * coef

df_local = pd.DataFrame({
    "Palavra": feature_names,
    "Valor": X_text_local.flatten(),
    "SHAP Value": vals
})
df_local["Abs"] = np.abs(df_local["SHAP Value"])
df_local = df_local[df_local["Valor"] > 0].sort_values("Abs", ascending=False)

# Top 10 positivas e negativas
df_pos = df_local[df_local["SHAP Value"] > 0].head(10)
df_neg = df_local[df_local["SHAP Value"] < 0].head(10)

print("\n=== Top 10 palavras explicativas para PRECONCEITO ===")
print(df_pos[["Palavra", "SHAP Value"]].to_string(index=False))

print("\n=== Top 10 palavras explicativas para NÃO PRECONCEITO ===")
print(df_neg[["Palavra", "SHAP Value"]].to_string(index=False))


# ==========================================
# 5. Fidelidade (Comprehensiveness / Sufficiency)
# ==========================================

tokens_exp = df_local.head(10)["Palavra"].tolist()

def compute_fidelity_nb(text, tokens_exp, model, lexicon):
    tokens_text = preprocess_data([text], experiment)[0].split()
    tokens_exp = [t for t in tokens_exp if t in tokens_text]

    palavras = list(lexicon.keys())
    idx_map = {p: i for i, p in enumerate(palavras)}

    def vectorize_tokens(tokens):
        features = np.zeros((1, len(palavras)))
        for t in tokens:
            if t in idx_map:
                features[0, idx_map[t]] += 1
        return csr_matrix(features)

    def prob_from_tokens(tokens):
        return model.predict_proba(vectorize_tokens(tokens))[0][1]

    p_original = prob_from_tokens(tokens_text)
    tokens_removed = [t for t in tokens_text if t not in tokens_exp]
    p_removed = prob_from_tokens(tokens_removed)

    p_only = prob_from_tokens(tokens_exp) if tokens_exp else 0.0

    comp = (p_original - p_removed) / max(p_original, 1e-6)
    suff = (p_original - p_only) / max(p_original, 1e-6)

    return comp, suff, p_original, p_removed, p_only


text_example = df.iloc[idx_local]["text_content_anonymous"]
comp, suff, p_o, p_r, p_s = compute_fidelity_nb(text_example, tokens_exp, last_model, lexicon)

print("\n=== Fidelidade (via MultinomialNB + Dicionário) ===")
print(f"Palavras explicativas: {tokens_exp}")
print(f"Probabilidade original: {p_o:.3f}")
print(f"Prob. sem palavras explicativas: {p_r:.3f}")
print(f"Prob. só com palavras explicativas: {p_s:.3f}")
print(f"Abrangência (Comprehensiveness): {comp:.3f}")
print(f"Suficiência (Sufficiency):       {suff:.3f}")


Rodando experimento: ml-dic-mnb
Pré-processamento padrão.
Tamanho total: 2997 | Classe 1: 1497 | Classe 0: 1500

=== Fold 1 ===
Acurácia: 0.723 | Precisão: 0.789 | Revocação: 0.610 | F1: 0.688 | AUC: 0.798

=== Fold 2 ===
Acurácia: 0.762 | Precisão: 0.840 | Revocação: 0.647 | F1: 0.731 | AUC: 0.827

=== Fold 3 ===
Acurácia: 0.721 | Precisão: 0.743 | Revocação: 0.676 | F1: 0.708 | AUC: 0.803

=== Fold 4 ===
Acurácia: 0.726 | Precisão: 0.773 | Revocação: 0.639 | F1: 0.700 | AUC: 0.795

=== Fold 5 ===
Acurácia: 0.743 | Precisão: 0.776 | Revocação: 0.682 | F1: 0.726 | AUC: 0.820

=== Resultados Médios (5 Folds) ===
Acurácia média: 0.735 ± 0.015
Precisão média: 0.784
Revocação média: 0.651
F1 média:        0.710
ROC AUC média:   0.809

=== SHAP LOCAL (MultinomialNB com Dicionário) ===

Amostra escolhida: índice 485 (contém 2 palavras do dicionário)

=== Top 10 palavras explicativas para PRECONCEITO ===
Palavra  SHAP Value
 judeus    1.391147

=== Top 10 palavras explicativas para NÃO PRECON

### LIME - MultinomialNB

In [14]:

# ==========================================
# 1. Preparação dos dados
# ==========================================

experiment = 'ml-dic-mnb'
print(f"Rodando experimento: {experiment}")

df = df.drop_duplicates(subset=['text_content_anonymous'])
df['text_content_anonymous'] = df['text_content_anonymous'].fillna("").astype(str)

pro_texts = preprocess_data(df, experiment)
y = df['preconceito']

print(f"Tamanho total: {len(y)} | Classe 1: {sum(y)} | Classe 0: {len(y) - sum(y)}")

# ==========================================
# 2. Validação cruzada
# ==========================================

kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
accs, precs, recs, f1s, aucs = [], [], [], [], []

fold = 1
last_model = None
last_X_test = None
feature_names = list(lexicon.keys())

for train_idx, test_idx in kf.split(pro_texts, y):
    X_train_texts = [pro_texts[i] for i in train_idx]
    X_test_texts = [pro_texts[i] for i in test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    # gerar matriz de features (dicionário)
    X_train, feature_names = gerar_features_dicionario(X_train_texts, lexicon)
    X_test, _ = gerar_features_dicionario(X_test_texts, lexicon)

    # === Modelo Multinomial Naive Bayes ===
    model = MultinomialNB()
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    accs.append(accuracy_score(y_test, y_pred))
    precs.append(precision_score(y_test, y_pred))
    recs.append(recall_score(y_test, y_pred))
    f1s.append(f1_score(y_test, y_pred))
    aucs.append(roc_auc_score(y_test, y_prob))

    print(f"\n=== Fold {fold} ===")
    print(f"Acurácia: {accs[-1]:.3f} | Precisão: {precs[-1]:.3f} | Revocação: {recs[-1]:.3f} | "
          f"F1: {f1s[-1]:.3f} | AUC: {aucs[-1]:.3f}")

    last_model = model
    last_X_test = X_test
    fold += 1

# ==========================================
# 3. Resultados médios
# ==========================================

print("\n=== Resultados Médios (5 Folds) ===")
print(f"Acurácia média: {np.mean(accs):.3f} ± {np.std(accs):.3f}")
print(f"Precisão média: {np.mean(precs):.3f}")
print(f"Revocação média: {np.mean(recs):.3f}")
print(f"F1 média:        {np.mean(f1s):.3f}")
print(f"ROC AUC média:   {np.mean(aucs):.3f}")

# ==========================================
# 4. Explicabilidade com LIME + Fidelidade
# ==========================================

class LexiconVectorizer:
    def __init__(self, lexicon):
        self.lexicon = lexicon
        self.vocab = list(lexicon.keys())
        self.idx_map = {palavra: i for i, palavra in enumerate(self.vocab)}

    def fit(self, textos, y=None):
        return self

    def transform(self, textos):
        features = np.zeros((len(textos), len(self.vocab)), dtype=int)
        for i, texto in enumerate(textos):
            for token in re.findall(r"\w+", texto.lower(), re.UNICODE):
                if token in self.idx_map:
                    features[i, self.idx_map[token]] += 1
        return features

    def get_feature_names_out(self):
        return np.array(self.vocab)


lex_vectorizer = LexiconVectorizer(lexicon)
pipeline = make_pipeline(lex_vectorizer, last_model)

class_names = ["Não preconceito", "Preconceito"]
explainer = LimeTextExplainer(class_names=class_names)

# Selecionar amostra com palavras do dicionário
nonzero_counts = np.array(last_X_test.getnnz(axis=1)).flatten()
nonzero_idx = np.where(nonzero_counts > 0)[0]

if len(nonzero_idx) == 0:
    print("Nenhuma amostra contém palavras do dicionário.")
else:
    idx_local_test = np.random.choice(nonzero_idx)
    real_idx = test_idx[idx_local_test]
    text_example = df.iloc[real_idx]["text_content_anonymous"]

    print(f"\n=== Texto selecionado (índice real {real_idx}) ===\n{text_example}\n")

    exp = explainer.explain_instance(
        text_instance=text_example,
        classifier_fn=pipeline.predict_proba,
        num_features=10
    )

    raw_exp = exp.as_list()
    lex_words_set = set([w.lower() for w in lexicon.keys()])
    rows = []
    for token, peso in raw_exp:
        m = re.search(r"\w+", token, re.UNICODE)
        tok_norm = m.group(0).lower() if m else token.lower().strip()
        if tok_norm in lex_words_set:
            rows.append((tok_norm, peso))

    if len(rows) == 0:
        print("Nenhuma palavra explicativa do dicionário foi identificada.")
    else:
        df_exp = pd.DataFrame(rows, columns=["Palavra", "Peso"])
        df_exp = df_exp.groupby("Palavra", as_index=False).agg({"Peso": "sum"})
        df_exp["Abs(Peso)"] = np.abs(df_exp["Peso"])
        df_exp = df_exp.sort_values(by="Abs(Peso)", ascending=False)

        exp_pos = df_exp[df_exp["Peso"] > 0].head(10)
        exp_neg = df_exp[df_exp["Peso"] < 0].head(10)

        print("\n=== Top 10 palavras explicativas para PRECONCEITO ===")
        print(exp_pos.to_string(index=False))
        print("\n=== Top 10 palavras explicativas para NÃO PRECONCEITO ===")
        print(exp_neg.to_string(index=False))

        tokens_exp = df_exp.head(10)["Palavra"].tolist()

        def compute_fidelity_lime(text, tokens_exp, model_pipeline):
            def prob(t):
                return model_pipeline.predict_proba([t])[0][1]

            p_original = prob(text)
            words = re.findall(r"\w+", text)
            text_removed = " ".join([w for w in words if w.lower() not in tokens_exp])
            text_only = " ".join([w for w in words if w.lower() in tokens_exp])

            p_removed = prob(text_removed) if text_removed.strip() else p_original
            p_only = prob(text_only) if text_only.strip() else 0

            comp = (p_original - p_removed) / max(abs(p_original), 1e-6)
            suff = (p_original - p_only) / max(abs(p_original), 1e-6)
            return comp, suff, p_original, p_removed, p_only

        comp, suff, p_o, p_r, p_s = compute_fidelity_lime(text_example, tokens_exp, pipeline)

        print("\n=== Fidelidade (via LIME + MultinomialNB + Dicionário) ===")
        print(f"Palavras explicativas: {tokens_exp}")
        print(f"Prob. original: {p_o:.3f}")
        print(f"Prob. sem palavras explicativas: {p_r:.3f}")
        print(f"Prob. só com palavras explicativas: {p_s:.3f}")
        print(f"Comprehensiveness: {comp:.3f}")
        print(f"Sufficiency:       {suff:.3f}")


Rodando experimento: ml-dic-mnb
Pré-processamento padrão.
Tamanho total: 2997 | Classe 1: 1497 | Classe 0: 1500

=== Fold 1 ===
Acurácia: 0.723 | Precisão: 0.789 | Revocação: 0.610 | F1: 0.688 | AUC: 0.798

=== Fold 2 ===
Acurácia: 0.762 | Precisão: 0.840 | Revocação: 0.647 | F1: 0.731 | AUC: 0.827

=== Fold 3 ===
Acurácia: 0.721 | Precisão: 0.743 | Revocação: 0.676 | F1: 0.708 | AUC: 0.803

=== Fold 4 ===
Acurácia: 0.726 | Precisão: 0.773 | Revocação: 0.639 | F1: 0.700 | AUC: 0.795

=== Fold 5 ===
Acurácia: 0.743 | Precisão: 0.776 | Revocação: 0.682 | F1: 0.726 | AUC: 0.820

=== Resultados Médios (5 Folds) ===
Acurácia média: 0.735 ± 0.015
Precisão média: 0.784
Revocação média: 0.651
F1 média:        0.710
ROC AUC média:   0.809

=== Texto selecionado (índice real 732) ===
EU VI ISSO NO OUTRO GRUPO ESTAVA FALANDO!
O Bolsonaro ta com outro massonico do AGRO que não querem concorrentes de fora da maçonaria e por isso estão criando essa propagando para o NOSSO AGRO QUE ESTA COM OS MAÇONS